# Microacciones: Efectividad por Estado Emocional

**Objetivo:** Identificar las mejores microacciones según el estado emocional previo del usuario

**Dataset:** 33 registros de 3 usuarios

**Método:** Normalización Z-score + Análisis de efectividad + Validación accuracy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# Cargar y procesar datos
df = pd.read_csv("datos_demo_luz/feedbacks_microacciones.csv")
print(f"Datos: {len(df)} registros, {df['usuario_id'].nunique()} usuarios, {df['microaccion'].nunique()} microacciones")

# Normalización Z-score por usuario
scaler = StandardScaler()
df_norm = df.copy()
cols = ['felicidad_previa', 'estres_previo', 'motivacion_previa', 'efectividad', 'comodidad', 'energia']

for usuario in df['usuario_id'].unique():
    mask = df_norm['usuario_id'] == usuario
    df_norm.loc[mask, cols] = scaler.fit_transform(df_norm.loc[mask, cols])

print("✓ Datos normalizados por usuario")
df.head()

In [ ]:
# ANÁLISIS: TOP microacciones y estados emocionales
print("🏆 RANKING DE MICROACCIONES POR EFECTIVIDAD")
print("="*50)

# Ranking de efectividad
ranking = df_norm.groupby('microaccion')['efectividad'].agg(['mean', 'count']).sort_values('mean', ascending=False)
ranking = ranking[ranking['count'] >= 2]

for i, (microaccion, data) in enumerate(ranking.head(5).iterrows(), 1):
    print(f"{i}. {microaccion.title()}: {data['mean']:+.2f} ({int(data['count'])} casos)")

# Análisis por estado emocional
print("\n📊 EFECTIVIDAD POR ESTADO EMOCIONAL")
print("="*50)

# Bins emocionales
df_norm['estres_alto'] = df_norm['estres_previo'] > 0
df_norm['felicidad_baja'] = df_norm['felicidad_previa'] < 0
df_norm['motivacion_baja'] = df_norm['motivacion_previa'] < 0

estados = {
    'Estrés Alto': df_norm[df_norm['estres_alto']],
    'Felicidad Baja': df_norm[df_norm['felicidad_baja']],
    'Motivación Baja': df_norm[df_norm['motivacion_baja']]
}

recomendaciones = {}
for estado, datos in estados.items():
    if len(datos) > 0:
        mejor = datos.groupby('microaccion')['efectividad'].mean().idxmax()
        efectividad = datos.groupby('microaccion')['efectividad'].mean().max()
        recomendaciones[estado] = (mejor, efectividad)
        print(f"{estado}: {mejor.title()} ({efectividad:+.2f})")

# VISUALIZACIÓN
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# TOP 5 microacciones
top_5 = ranking.head(5)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_5)))
bars = axes[0].barh(top_5.index, top_5['mean'], color=colors)
axes[0].set_title('TOP 5 Microacciones Más Efectivas', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Efectividad (Z-score)')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.7)

for bar, value in zip(bars, top_5['mean']):
    axes[0].text(value + 0.02 if value >= 0 else value - 0.02, bar.get_y() + bar.get_height()/2, 
                f'{value:.2f}', va='center', ha='left' if value >= 0 else 'right', fontweight='bold')

# Recomendaciones por estado
estados_list = list(recomendaciones.keys())
efectividades = [recomendaciones[estado][1] for estado in estados_list]
microacciones_recom = [recomendaciones[estado][0] for estado in estados_list]

bars = axes[1].bar(estados_list, efectividades, color=['#ff6b6b', '#4ecdc4', '#feca57'], alpha=0.8)
axes[1].set_title('Mejores Microacciones por Estado', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Efectividad')
axes[1].set_xticklabels(estados_list, rotation=45, ha='right')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)

for i, (bar, value, microaccion) in enumerate(zip(bars, efectividades, microacciones_recom)):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{microaccion}\n{value:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# VALIDACIÓN: Accuracy del modelo
top_3 = list(ranking.head(3).index)

# Preparar datos para modelo
X = df_norm[['felicidad_previa', 'estres_previo', 'motivacion_previa']]
y = (df_norm['efectividad'] > 0).astype(int)

# Entrenar modelo
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X, y)
accuracy = model.score(X, y) * 100

# Comparar TOP vs otras
efectividad_top = df_norm[df_norm['microaccion'].isin(top_3)]['efectividad'].mean()
efectividad_otras = df_norm[~df_norm['microaccion'].isin(top_3)]['efectividad'].mean()
ventaja = efectividad_top - efectividad_otras

print("🎯 VALIDACIÓN DEL MODELO")
print("="*40)
print(f"Accuracy general: {accuracy:.1f}%")
print(f"TOP 3 efectividad: {efectividad_top:+.2f}")
print(f"Otras efectividad: {efectividad_otras:+.2f}")
print(f"Ventaja TOP 3: +{ventaja:.2f} puntos")

# Medidor visual
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
theta = np.linspace(0, np.pi, 100)
x = np.cos(theta)
y_circle = np.sin(theta)
ax.plot(x, y_circle, 'k-', linewidth=4)

angle = np.pi * (1 - accuracy/100)
needle_x = 0.8 * np.cos(angle)
needle_y = 0.8 * np.sin(angle)
color = 'green' if accuracy >= 70 else 'orange' if accuracy >= 50 else 'red'
ax.arrow(0, 0, needle_x, needle_y, head_width=0.08, head_length=0.08, 
         fc=color, ec='black', linewidth=3)

ax.set_xlim(-1.2, 1.2)
ax.set_ylim(0, 1.3)
ax.set_aspect('equal')
ax.set_title('Accuracy del Modelo', fontsize=16, fontweight='bold')
ax.text(0, -0.3, f'{accuracy:.1f}%', ha='center', va='center', 
        fontsize=24, fontweight='bold', color=color)
ax.axis('off')

plt.show()

if accuracy >= 70:
    print("✅ EXCELENTE - Modelo muy confiable")
elif accuracy >= 50:
    print("✅ BUENO - Modelo útil")
else:
    print("⚠️ MEJORAR - Necesita más datos")

In [ ]:
# CONCLUSIONES FINALES
print("📋 RECOMENDACIONES BASADAS EN EVIDENCIA")
print("="*60)

print("\n🏆 TOP 3 MICROACCIONES GENERALES:")
for i, (microaccion, data) in enumerate(ranking.head(3).iterrows(), 1):
    print(f"  {i}. {microaccion.title()} → {data['mean']:+.2f} efectividad")

print("\n🎯 RECOMENDACIONES POR ESTADO EMOCIONAL:")
emojis = ['😰', '😔', '😕']
for i, (estado, (microaccion, efectividad)) in enumerate(recomendaciones.items()):
    print(f"  {emojis[i]} {estado}: {microaccion.title()} ({efectividad:+.2f})")

print(f"\n📊 MÉTRICAS DEL MODELO:")
print(f"  • Accuracy: {accuracy:.1f}%")
print(f"  • Ventaja TOP 3: +{ventaja:.2f} puntos")
print(f"  • Dataset: {len(df)} registros, {df['usuario_id'].nunique()} usuarios")

print(f"\n✅ MODELO VALIDADO - LISTO PARA IMPLEMENTACIÓN")